<a href="https://colab.research.google.com/github/yicheng-kang/team10-nbashots-sports/blob/main/CS_131_Spring26__20__week_11__Tuesday__When_to_move_beyond_bash.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CS 131 - 20 - PySpark Hands-On Notebook  
## When to Move Beyond Bash + PySpark Intro (Local Mode)

Goal:

1. create a Spark session  
2. download a real dataset  
3. read Parquet and CSV files  
4. build a simple PySpark pipeline  
5. write results back to disk  
6. inspect the execution plan

---

## Learning goals:

- explain why Spark is useful when Bash pipelines become hard to manage
- read data into PySpark DataFrames
- tell the difference between **transformations** and **actions**
- use `select`, `filter`, `withColumn`, `join`, `groupBy`, and `agg`
- write results as Parquet
- recognize a few important pieces of a Spark execution plan

## Before you start

This notebook is designed to work in:

- **Google Colab**
- **Jupyter Notebook/JupyterLab**
- another Python notebook environment with Java available

If `pyspark` is not installed, the next cell installs it.

In [ ]:
import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pyspark") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
    print("Installed pyspark.")
else:
    print("pyspark is already available.")

pyspark is already available.


## 1. Create a Spark session

A **SparkSession** is the main entry point for PySpark.  
For this class, we run Spark in **local mode**, which means Spark uses the CPU cores on your own machine.

`local[*]` means: use all available local CPU cores.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CS131-Class20-PySpark-Intro")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")   # keep it small for laptops
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

Spark version: 4.0.2
Shuffle partitions: 8


## 2. Import the functions we will use


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

from pyspark.sql.functions import (
    col,
    to_date,
    avg,
    count,
    broadcast
)

## 3. Download the data

We will use two files from the NYC Taxi dataset:

- **yellow taxi trip data** in **Parquet**
- **taxi zone lookup** in **CSV**

This small example lets us practice:

- reading two different file formats
- joining a large table with a small lookup table
- aggregating the result

In [ ]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

files_to_download = {
    "yellow_tripdata_2022-01.parquet": "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2022-01.parquet",
    "taxi_zone_lookup.csv": "https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv",
}

for filename, url in files_to_download.items():
    target = data_dir/filename
    if not target.exists():
        print(f"Downloading {filename} ...")
        urlretrieve(url, target)
    else:
        print(f"{filename} already exists.")

for path in sorted(data_dir.iterdir()):
    size_mb = path.stat().st_size / (1024 * 1024)
    print(f"{path.name:30s} {size_mb:8.2f} MB")

yellow_tripdata_2022-01.parquet already exists.
taxi_zone_lookup.csv already exists.
taxi_zone_lookup.csv               0.01 MB
yellow_tripdata_2022-01.parquet    36.37 MB


### Checkpoint

Before reading the files, notice:

- the trip data is stored as **Parquet**
- the zone lookup is stored as **CSV**

In the following sections: pay attention to how much easier it is for Spark to work with typed Parquet data than with raw CSV text.

## 4. Read the data into DataFrames

A **DataFrame** is a table with named columns and data types.

We will:

- read the Parquet file into `trips`
- read the CSV lookup file into `zones`
- inspect the schema and a few rows

In [ ]:
# Read Parquet: types are already stored in the file
trips = spark.read.parquet(str(data_dir/"yellow_tripdata_2022-01.parquet"))

# Read CSV: need header=True and inferSchema=True
zones = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(data_dir/"taxi_zone_lookup.csv"))
)

print("Trips schema:")
trips.printSchema()

print("Zones schema:")
zones.printSchema()

Trips schema:
root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)

Zones schema:
root
 |-- LocationID: integer (nullable = true)
 |-- Borough: string (nullable = true)
 |-- Zone: stri

In [ ]:
print("Trips sample:")
trips.show(5, truncate=False)

print("Zones sample:")
zones.show(5, truncate=False)

Trips sample:
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|1       |2022-01-01 00:35:40 |2022-01-01 00:53:29  |2.0            |3.8          |1.0       |N                 |142         |236         |1           |14.5       |3.0  |0.5

In [ ]:
trip_count = trips.count()
zone_count = zones.count()

print(f"Number of trip rows:  {trip_count:,}")
print(f"Number of zone rows:  {zone_count:,}")

Number of trip rows:  2,463,931
Number of zone rows:  265


## 5. First look at transformations vs actions

In PySpark, many commands are **transformations**.  
That means they describe what you want, but they do not run immediately.

Examples:

- `select(...)`
- `filter(...)`
- `withColumn(...)`
- `join(...)`
- `groupBy(...).agg(...)`

An **action** is what triggers Spark to actually do the work.

Examples:

- `show()`
- `count()`
- `collect()`
- `write.parquet(...)`

In the next step, we will build a pipeline with several transformations first.

## 6. Build a cleaner trips table

We do not need every column. A common good habit is:

1. **select early**
2. **filter early**
3. create only the derived columns you need

This helps reduce I/O and makes the pipeline easier to understand.

In [ ]:
clean = (
    trips
    .select(
        "tpep_pickup_datetime",
        "PULocationID",
        "passenger_count",
        "total_amount"
    )
    .filter(
        (col("passenger_count") >= 1) &
        (col("total_amount").between(1, 200))
    )
    .withColumn("day", to_date(col("tpep_pickup_datetime")))
)

# No action has happened yet just because we defined 'clean'
clean

DataFrame[tpep_pickup_datetime: timestamp_ntz, PULocationID: bigint, passenger_count: double, total_amount: double, day: date]

### Why these filters?

It is often helpful to remove obviously strange values, eg.:

- `passenger_count >= 1`
- `total_amount` between 1 and 200

These are not perfect business rules.  
They are simple cleaning choices so the downstream aggregation is easier to interpret.

In [ ]:
print("Schema after selecting/filtering/adding 'day':")
clean.printSchema()

print("Preview:")
clean.show(5, truncate=False)

Schema after selecting/filtering/adding 'day':
root
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- day: date (nullable = true)

Preview:
+--------------------+------------+---------------+------------+----------+
|tpep_pickup_datetime|PULocationID|passenger_count|total_amount|day       |
+--------------------+------------+---------------+------------+----------+
|2022-01-01 00:35:40 |142         |2.0            |21.95       |2022-01-01|
|2022-01-01 00:33:43 |236         |1.0            |13.3        |2022-01-01|
|2022-01-01 00:53:21 |166         |1.0            |10.56       |2022-01-01|
|2022-01-01 00:25:21 |114         |1.0            |11.8        |2022-01-01|
|2022-01-01 00:36:48 |68          |1.0            |30.3        |2022-01-01|
+--------------------+------------+---------------+------------+----------+
only showing top 5 rows


## 7. Join with the taxi zone lookup

`PULocationID` is a numeric location identifier.  
The lookup table tells us which **Borough** and **Zone** each ID refers to.

The lookup table is small, so we use `broadcast(zones)`.

That tells Spark:

> this table is small enough to copy to each worker/task instead of shuffling both sides of the join

That often makes small-dimension joins much faster.

In [ ]:
joined = clean.join(
    broadcast(zones),
    col("PULocationID") == col("LocationID"),
    how="inner"
)

joined.show(5, truncate=False)

+--------------------+------------+---------------+------------+----------+----------+---------+-----------------------+------------+
|tpep_pickup_datetime|PULocationID|passenger_count|total_amount|day       |LocationID|Borough  |Zone                   |service_zone|
+--------------------+------------+---------------+------------+----------+----------+---------+-----------------------+------------+
|2022-01-01 00:35:40 |142         |2.0            |21.95       |2022-01-01|142       |Manhattan|Lincoln Square East    |Yellow Zone |
|2022-01-01 00:33:43 |236         |1.0            |13.3        |2022-01-01|236       |Manhattan|Upper East Side North  |Yellow Zone |
|2022-01-01 00:53:21 |166         |1.0            |10.56       |2022-01-01|166       |Manhattan|Morningside Heights    |Boro Zone   |
|2022-01-01 00:25:21 |114         |1.0            |11.8        |2022-01-01|114       |Manhattan|Greenwich Village South|Yellow Zone |
|2022-01-01 00:36:48 |68          |1.0            |30.3       

## 8. Aggregate: daily trips by borough

Now we group by:

- `Borough`
- `day`

and compute:

- number of trips
- average total fare

In [ ]:
daily = (
    joined
    .groupBy("Borough", "day")
    .agg(
        count("*").alias("trips"),
        avg("total_amount").alias("avg_fare")
    )
    .orderBy("day", "Borough")
)

daily.show(15, truncate=False)

+-------------+----------+-----+------------------+
|Borough      |day       |trips|avg_fare          |
+-------------+----------+-----+------------------+
|Manhattan    |2008-12-31|4    |11.715000000000002|
|Queens       |2008-12-31|2    |49.98             |
|Manhattan    |2009-01-01|7    |27.621428571428577|
|Manhattan    |2021-12-31|22   |15.765000000000004|
|Queens       |2021-12-31|2    |45.335            |
|Bronx        |2022-01-01|75   |25.551599999999983|
|Brooklyn     |2022-01-01|476  |25.689642857142793|
|EWR          |2022-01-01|41   |87.8563414634147  |
|Manhattan    |2022-01-01|52460|17.31396797560819 |
|N/A          |2022-01-01|59   |74.40711864406782 |
|Queens       |2022-01-01|5980 |50.49187123745673 |
|Staten Island|2022-01-01|1    |74.55             |
|Unknown      |2022-01-01|580  |29.081517241379213|
|Bronx        |2022-01-02|67   |30.713134328358198|
|Brooklyn     |2022-01-02|240  |27.448583333333392|
+-------------+----------+-----+------------------+
only showing

### Read this pipeline from left to right

- read trip data
- keep only a few relevant columns
- filter out rows we do not want
- create a day column from the timestamp
- join to the zone lookup
- group by borough and day
- compute trip counts and average fare
- sort the final result

If you can explain it clearly in words, you usually understand the code better too.

In [ ]:
daily2 = ( spark.read.parquet(str(data_dir/"yellow_tripdata_2022-01.parquet"))
    .select(
        "tpep_pickup_datetime",
        "PULocationID",
        "passenger_count",
        "total_amount"
    )
    .filter(
        (col("passenger_count") >= 1) &
        (col("total_amount").between(1, 200))
    )
    .withColumn("day", to_date(col("tpep_pickup_datetime")))
    .join(
      broadcast( zones ),
      col("PULocationID") == col("LocationID"),
      how="inner"
    ).groupBy("Borough", "day")
    .agg(
        count("*").alias("trips"),
        avg("total_amount").alias("avg_fare")
    )
    .orderBy("day", "Borough")
)


In [ ]:
daily2.show(15, truncate=False)

+-------------+----------+-----+------------------+
|Borough      |day       |trips|avg_fare          |
+-------------+----------+-----+------------------+
|Manhattan    |2008-12-31|4    |11.715000000000002|
|Queens       |2008-12-31|2    |49.98             |
|Manhattan    |2009-01-01|7    |27.621428571428577|
|Manhattan    |2021-12-31|22   |15.765000000000004|
|Queens       |2021-12-31|2    |45.335            |
|Bronx        |2022-01-01|75   |25.551599999999983|
|Brooklyn     |2022-01-01|476  |25.689642857142793|
|EWR          |2022-01-01|41   |87.8563414634147  |
|Manhattan    |2022-01-01|52460|17.31396797560819 |
|N/A          |2022-01-01|59   |74.40711864406782 |
|Queens       |2022-01-01|5980 |50.49187123745673 |
|Staten Island|2022-01-01|1    |74.55             |
|Unknown      |2022-01-01|580  |29.081517241379213|
|Bronx        |2022-01-02|67   |30.713134328358198|
|Brooklyn     |2022-01-02|240  |27.448583333333392|
+-------------+----------+-----+------------------+
only showing

## 9. Write the result as Parquet

This is an important pattern in real data work:

- raw input may come as CSV or logs
- intermediate and final analytic tables are often better stored as **Parquet**

Why?

- typed schema
- better compression
- columnar layout
- better reuse later

In [ ]:
output_dir = Path("output")/"daily_by_borough"

daily.write.mode("overwrite").parquet(str(output_dir))
print(f"Wrote results to: {output_dir}")

Wrote results to: output/daily_by_borough


In [ ]:
daily_saved = spark.read.parquet(str(output_dir))
daily_saved.show(10, truncate=False)

+---------+----------+-----+------------------+
|Borough  |day       |trips|avg_fare          |
+---------+----------+-----+------------------+
|Manhattan|2008-12-31|4    |11.715000000000002|
|Queens   |2008-12-31|2    |49.98             |
|Manhattan|2009-01-01|7    |27.621428571428577|
|Manhattan|2021-12-31|22   |15.765000000000004|
|Queens   |2021-12-31|2    |45.335            |
|Bronx    |2022-01-01|75   |25.551599999999983|
|Brooklyn |2022-01-01|476  |25.689642857142793|
|EWR      |2022-01-01|41   |87.8563414634147  |
|Manhattan|2022-01-01|52460|17.31396797560819 |
|N/A      |2022-01-01|59   |74.40711864406782 |
+---------+----------+-----+------------------+
only showing top 10 rows


In [ ]:
def folder_size_mb(path: Path) -> float:
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / (1024 * 1024)

input_size_mb = (data_dir/"yellow_tripdata_2022-01.parquet").stat().st_size / (1024 * 1024)
output_size_mb = folder_size_mb(output_dir)

print(f"Input Parquet size:  {input_size_mb:8.2f} MB")
print(f"Output folder size:  {output_size_mb:8.2f} MB")

Input Parquet size:     36.37 MB
Output folder size:      0.00 MB


## 10. Inspect the execution plan

One of the most important PySpark habits is learning to read `explain()` output.

You do **not** need to understand every line yet.

For today, look for these ideas:

- **FileScan parquet** --> Spark is reading Parquet files
- **BroadcastHashJoin** --> the small lookup table was broadcast
- **HashAggregate** --> Spark is doing the aggregation
- **Exchange** --> data was shuffled between partitions

In [ ]:
daily.explain(True)

== Parsed Logical Plan ==
'Sort ['day ASC NULLS FIRST, 'Borough ASC NULLS FIRST], true
+- Aggregate [Borough#77, day#188], [Borough#77, day#188, count(1) AS trips#237L, avg(total_amount#56) AS avg_fare#238]
   +- Join Inner, (PULocationID#47L = cast(LocationID#76 as bigint))
      :- Project [tpep_pickup_datetime#41, PULocationID#47L, passenger_count#43, total_amount#56, to_date(tpep_pickup_datetime#41, None, Some(Etc/UTC), true) AS day#188]
      :  +- Filter ((passenger_count#43 >= cast(1 as double)) AND ((total_amount#56 >= cast(1 as double)) AND (total_amount#56 <= cast(200 as double))))
      :     +- Project [tpep_pickup_datetime#41, PULocationID#47L, passenger_count#43, total_amount#56]
      :        +- Relation [VendorID#40L,tpep_pickup_datetime#41,tpep_dropoff_datetime#42,passenger_count#43,trip_distance#44,RatecodeID#45,store_and_fwd_flag#46,PULocationID#47L,DOLocationID#48L,payment_type#49L,fare_amount#50,extra#51,mta_tax#52,tip_amount#53,tolls_amount#54,improvement_surchar

## 11. Mini exercises

Do the exercises below.  
They are meant to help you move from copying code to modifying a pipeline yourself.

### Exercise A
Create a table with:

- `day`
- total number of trips for that day across **all** boroughs

Call the result `daily_total_trips`.

In [ ]:
# Your turn:
daily_total_trips = (
    ...
)

daily_total_trips.show()

+----------+-----+
|       day|trips|
+----------+-----+
|2008-12-31|    6|
|2009-01-01|    7|
|2021-12-31|   24|
|2022-01-01|59672|
|2022-01-02|55662|
|2022-01-03|68952|
|2022-01-04|71075|
|2022-01-05|70951|
|2022-01-06|76068|
|2022-01-07|67674|
|2022-01-08|79238|
|2022-01-09|60955|
|2022-01-10|69913|
|2022-01-11|73511|
|2022-01-12|76320|
|2022-01-13|80786|
|2022-01-14|88830|
|2022-01-15|83758|
|2022-01-16|68426|
|2022-01-17|60227|
+----------+-----+
only showing top 20 rows


### Exercise B
Find the top 10 borough/day combinations by number of trips.

Hint: sort by the `trips` column in descending order.

In [ ]:
# Your turn:
top_days = ...

top_days.show(10, truncate=False)

+---------+----------+-----+------------------+
|Borough  |day       |trips|avg_fare          |
+---------+----------+-----+------------------+
|Manhattan|2022-01-21|88513|16.191774202670175|
|Manhattan|2022-01-27|87458|17.03487765557142 |
|Manhattan|2022-01-26|85539|16.269727492732898|
|Manhattan|2022-01-22|85231|15.951976510904027|
|Manhattan|2022-01-28|82855|17.308482529731638|
|Manhattan|2022-01-14|81467|16.385586433780986|
|Manhattan|2022-01-20|79686|16.115023969088163|
|Manhattan|2022-01-15|77062|15.742761153366303|
|Manhattan|2022-01-25|76860|16.163495966702023|
|Manhattan|2022-01-19|76172|16.277020821308597|
+---------+----------+-----+------------------+
only showing top 10 rows


### Exercise C
Filter the joined table to only rows where the borough is `"Manhattan"`, then compute the average fare by day.

Call the result `manhattan_daily`.

In [ ]:
# Your turn:
manhattan_daily = (
    ...
)

manhattan_daily.show(10, truncate=False)

+----------+------------------+
|day       |avg_fare          |
+----------+------------------+
|2008-12-31|11.715000000000002|
|2009-01-01|27.621428571428577|
|2021-12-31|15.765000000000004|
|2022-01-01|17.31396797560819 |
|2022-01-02|17.279360789272374|
|2022-01-03|16.00272473267316 |
|2022-01-04|15.903621729088858|
|2022-01-05|15.900646180728112|
|2022-01-06|15.958679771232198|
|2022-01-07|15.582849193737953|
+----------+------------------+
only showing top 10 rows


## 12. Reflection questions

Write short answers in your own words.

1. Why is Parquet usually better than CSV for repeated analytic work?
2. What is the benefit of selecting only the columns you need early in the pipeline?
3. Why might a broadcast join be a good idea for the zone lookup table?
4. Which commands in this notebook were **actions**?
5. Which commands were just **transformations** until an action happened?

## 13. Optional extension

Stretch goal:

- add `hour` using the pickup timestamp
- compute trip counts by `Borough` and `hour`
- find the busiest pickup hour
- calculate revenue per borough
-  compute average tip_amount, average fare_amount, and average tip rate defined as tip_amount/fare_amount
- compare weekdays vs weekends
- try `daily.explain(True)` again after changing the pipeline

## 14. Clean up

If you are done, you can stop the Spark session.

In [ ]:
# Uncomment when you are completely done:
# spark.stop()